# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. We will load metadata and records defined by the [Croissant schema](https://mlcommons.org/croissant/) and walk through data extraction, processing, and visualization steps suitable for scientific and machine learning applications.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library and dependencies are installed
!pip install mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields using their `@id` values.

In [ ]:
# List all record sets and their information
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s).\n")

record_set_ids = []
for rs in record_sets:
    print(f"Record set: {rs.name} (@id: {rs.id})")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else '(no description)' }")
    field_ids = [f.id for f in rs.fields]
    print(f"  Fields ({len(field_ids)}):")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()
    record_set_ids.append(rs.id)

# For demonstration, show one example record from each record set
for rs in record_sets:
    print(f"Example record from record set '{rs.name}' (@id: {rs.id}):")
    records = list(dataset.records(record_set=rs.id))
    if len(records) > 0:
        # Show only the first record
        print(json.dumps(records[0], indent=2))
    else:
        print("  No records available.")
    print("-"*60)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Refer to entities by their `@id` fields.

In [ ]:
# Extract all records for each record set using their @id
# Store as Pandas DataFrames for ease of use

dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Record set @id: {rs_id} - Shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        print()
    else:
        print(f"Record set @id: {rs_id} has no records.")

# For illustration, select the primary tabular record set if present
main_rs_id = None
max_cols = 0
for rs_id, df in dataframes.items():
    if df.shape[1] > max_cols:
        main_rs_id = rs_id
        max_cols = df.shape[1]

if main_rs_id is not None:
    print(f"Main tabular record set inferred as @id: {main_rs_id}\n")
    print(dataframes[main_rs_id].head())
else:
    print("No record set with tabular data found.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing such as filtering, normalization, and grouping. All fields referenced are via their `@id` values.

In [ ]:
# EDA: Filter, normalize, and group by field using @id references only
# Choose a numeric field using its @id

if main_rs_id is not None:
    df = dataframes[main_rs_id].copy()

    print(f"Fields in main DataFrame:")
    for i, col in enumerate(df.columns):
        print(f"{i}: {col}")

    # Try to select an integer or float field (choose by inspection or heuristics if not documented)
    # For this example, let's try '@age', '@interval_months', '@diagnosis_interval', etc.
    # Let's auto-select the first numeric-looking field
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
        # Try to cast to numeric if possible
        try:
            pd.to_numeric(df[col])
            numeric_field_id = col
            break
        except Exception:
            pass

    if numeric_field_id is not None:
        # Ensure numeric type
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].quantile(0.75)  # e.g., filter above 75th percentile
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold} (top quartile):")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to group by a categorical field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() < df.shape[0] // 2:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nMean {numeric_field_id} grouped by {group_field_id} (for filtered data):")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No main record set DataFrame to analyze.")

## 5. Visualization
Visualize distributions or relationships between key fields. Use field `@id` as labels.

Below, we display a histogram and a boxplot of the selected numeric field, and a bar plot if grouping field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id is not None and 'numeric_field_id' in locals() and numeric_field_id is not None:
    plt.figure(figsize=(10,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    plt.figure(figsize=(6,4))
    sns.boxplot(y=df[numeric_field_id].dropna())
    plt.title(f'Boxplot of {numeric_field_id}')
    plt.ylabel(numeric_field_id)
    plt.show()

    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,5))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No fields/columns suitable for visualization found.')

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and examine the Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors dataset through its Croissant schema. 

- We explored the record sets and fields using their `@id`s, ensuring robust and future-proof referencing.
- We extracted records into Pandas DataFrames for flexible analysis.
- We demonstrated filtering and normalization on numeric `@id` fields and showed how to group and visualize data distributions, using only `@id` field references in all processing.

This workflow provides a reproducible, standards-based method for data exploration in biomedical informatics and beyond.